## Lesson contract: Retrieval strategies

**Scenario.** Compare lexical, dense-like, and hybrid evidence for an incident. This notebook is the primary lesson: the theory, design trade-offs, runnable implementation, experiment, and reflection live together. The examples are deterministic so they run without credentials; replace the adapter boundary with a hosted model or vector service only after the behavior is tested.

### Core idea

A reliable RAG system separates evidence acquisition from answer generation. It preserves source identity, metadata, and a trace of decisions. Retrieval can fail because the corpus is incomplete, stale, unauthorized, or ambiguous; therefore a production answer needs a confidence policy and an explicit abstention path.

### Architecture map

```text
FLOW (read top to bottom)

+--------------------+
| Question           |
+--------------------+
          |
          v
+--------------------+
| Retrieve evidence  |
+--------------------+
          |
          v
+--------------------+
| Filter and rank    |
+--------------------+
          |
          v
+--------------------+
| Bounded context    |
+--------------------+
          |
          v
+------------------------+
| Generate or synthesize |
+------------------------+
          |
          v
+--------------------+
| Supported?         |
+--------------------+

Decision branches:
  +-- yes --> [Cited answer]
  `-- no --> [Abstain or recover]
```

Before running code, write down which state crosses each arrow and which component is allowed to make a model decision.

In [ ]:
from dataclasses import dataclass
from collections import Counter
import re

@dataclass(frozen=True)
class Evidence:
    doc_id: str
    text: str
    source: str
    metadata: dict

CORPUS = [
    Evidence("runbook-17", "Correlate checkout errors with the 08:42 deployment before proposing rollback.", "runbooks/checkout.md", {"tenant":"acme","team":"payments","version":3}),
    Evidence("incident-22", "A European checkout timeout followed a dependency latency spike; no records were lost.", "incidents/22.md", {"tenant":"acme","team":"payments","version":5}),
    Evidence("policy-04", "Enterprise customers receive a status update within 30 minutes of a confirmed incident.", "policies/sla.md", {"tenant":"acme","team":"support","version":7}),
]

def terms(text):
    return re.findall(r"[a-z0-9]+", text.lower())

def retrieve(query, k=2, tenant="acme"):
    q=Counter(terms(query))
    allowed=[d for d in CORPUS if d.metadata.get("tenant")==tenant]
    return sorted(allowed, key=lambda d: sum((q & Counter(terms(d.text))).values()), reverse=True)[:k]

question="Checkout is timing out in Europe after a deployment. What should support do?"
hits=retrieve(question)
[(h.doc_id,h.source,h.metadata) for h in hits]

### Example 1 — inspect and cite evidence

Do not pass opaque strings to a model. The context builder keeps stable IDs, source paths, versions, and tenant metadata so a reviewer can reproduce the answer and an authorization layer can audit inclusion. Remove one field and discuss which guarantee disappears.

In [ ]:
def build_context(hits):
    return "\n".join(f"[{h.doc_id}] {h.text} (source={h.source}; version={h.metadata['version']})" for h in hits)

context=build_context(hits)
answer="Investigate dependency latency and correlate it with the deployment before proposing rollback. If confirmed, update enterprise customers within 30 minutes."
print(context)
print("\nANSWER:", answer)

### Example 2 — insufficient evidence and failure recovery

The second query is intentionally unrelated. A safe system does not convert a plausible retrieved sentence into an answer. In a real implementation, replace this small lexical check with an evaluated relevance/groundedness grader, but keep the same explicit statuses: `answer`, `recover`, and `abstain`.

In [ ]:
def answer_question(query, k=2):
    selected=retrieve(query,k=k)
    evidence=" ".join(d.text.lower() for d in selected)
    required=("checkout" in query.lower() and "deployment" in evidence)
    if not selected or not required:
        return {"status":"abstain","reason":"evidence is missing or not specific enough","citations":[d.doc_id for d in selected]}
    return {"status":"answer","text":answer,"citations":[d.doc_id for d in selected]}

print(answer_question(question))
print(answer_question("Which planets have rings?"))

### Experiment and production checklist

Run the next cell with different `k` values. Record source IDs, context length, and whether the answer remains supported. Then add a stale document and a different tenant; prove they are filtered or trigger abstention.

Production checklist:

- filter authorization and tenant metadata before context assembly;
- keep original query, rewrites, candidates, and scores in a trace;
- cap context size, retries, tool calls, latency, and spend;
- monitor freshness, retrieval recall, citation coverage, abstention rate, and answer quality;
- retain a kill switch and a rollbackable index/prompt version.

### Practice

1. Add a fourth document about Payments and test a false positive.
2. Add `effective_at` and reject stale runbooks.
3. Replace the lexical scorer with a `Retriever` protocol and document where a dense or hybrid implementation fits.
4. Write one regression test for empty, contradictory, and unauthorized evidence.

Reference: https://qdrant.tech/documentation/concepts/hybrid-queries/

In [ ]:
for k in (1,2,3):
    selected=retrieve(question,k=k)
    print({"k":k,"ids":[d.doc_id for d in selected],"context_chars":len(build_context(selected))})

# Retrieval strategies

Retrieval is an information-retrieval problem inside a RAG system. This notebook compares an inspectable BM25 implementation with a simulated dense ranking and combines them with reciprocal-rank fusion.

## Why multiple signals?

Lexical retrieval is strong for exact identifiers and dense retrieval is often better at paraphrases. Raw scores from different systems are not directly comparable, so rank fusion is a practical baseline.

```text
FLOW (read top to bottom)

+--------------------+
| Query              |
+--------------------+

Decision branches:
  +-- next --> [BM25]
  `-- next --> [Dense adapter]

Supporting paths:
  [BM25] --feeds--> [RRF fusion]
  [Dense adapter] --feeds--> [RRF fusion]
```

In [ ]:
from src.rag_core.retrieval import BM25, Document, reciprocal_rank_fusion

documents = [
    Document('auth', 'API keys use the Authorization header and can be rotated.'),
    Document('errors', 'Error E401 means the request is unauthorized.'),
    Document('billing', 'Invoices are available from the billing endpoint.'),
]
bm25 = BM25(documents)
lexical = bm25.search('unauthorized API request', top_k=3)
[(doc.doc_id, round(score, 3)) for doc, score in lexical]

In [ ]:
# A dense retriever would return Documents in this shape.
dense = [documents[1], documents[0], documents[2]]
hybrid = reciprocal_rank_fusion([doc for doc, _ in lexical], dense)
[(doc.doc_id, round(score, 4)) for doc, score in hybrid]

## 1. Filter before every retrieval signal

A lexical or dense ranking must never see documents outside the caller's scope. The small document contract below includes tenant and freshness metadata. This is an authorization and data-governance boundary, not a model prompt.


In [ ]:
from src.rag_core.retrieval import (
    AttributedDocument, BM25, Document, filter_documents, hybrid_retrieve, ranking_metrics,
    static_dense_ranking, weighted_reciprocal_rank_fusion,
)

OPS_DOCS = [
    AttributedDocument('acme-runbook', 'E401 means unauthorized. Verify the API token before retrying the checkout request.', {'tenant': 'acme', 'freshness': 'current'}),
    AttributedDocument('acme-incident', 'European checkout timeouts followed the 08:42 deployment. Link the status page and avoid promising recovery time.', {'tenant': 'acme', 'freshness': 'current'}),
    AttributedDocument('acme-sla', 'Enterprise customers receive an update every 30 minutes while a confirmed incident remains open.', {'tenant': 'acme', 'freshness': 'current'}),
    AttributedDocument('globex-secret', 'Globex incident notes: E401 was caused by a private certificate rotation.', {'tenant': 'globex', 'freshness': 'current'}),
    AttributedDocument('acme-stale', 'Legacy policy: enterprise customers receive an update every 60 minutes.', {'tenant': 'acme', 'freshness': 'stale'}),
]
visible = filter_documents(OPS_DOCS, {'tenant': 'acme', 'freshness': 'current'})
[(doc.doc_id, doc.metadata) for doc in visible]


## 2. Compare exact identifiers and paraphrases

BM25 is a valuable baseline for exact identifiers such as `E401`. The deterministic dense adapter below stands in for externally computed embedding scores; it makes the interface and evaluation contract visible without downloading a model. In production replace this adapter with a retrieval-trained query/document encoder, retaining filters, IDs, budgets, and tests.


In [ ]:
exact_query = 'What does E401 mean?'
paraphrase_query = 'Why is the checkout request being rejected?'
dense_scores = {'acme-runbook': 0.86, 'acme-incident': 0.61, 'acme-sla': 0.08}

for query in (exact_query, paraphrase_query):
    lexical_docs = [doc for doc, _ in BM25(visible).search(query, top_k=3)]
    dense_docs = static_dense_ranking(visible, dense_scores, top_k=3)
    print(f'\n{query}')
    print('BM25:', [doc.doc_id for doc in lexical_docs])
    print('dense adapter:', [doc.doc_id for doc in dense_docs])


## 3. Fuse ranks and rerank only candidates

BM25 and dense scores are not on a shared scale. Reciprocal-rank fusion uses positions rather than raw scores. It can make a robust hybrid baseline when one signal catches identifiers and another catches paraphrases. A reranker then reorders only the bounded fused candidate set; it cannot recover a document neither first-stage retriever returned.


In [ ]:
results, trace = hybrid_retrieve(
    paraphrase_query,
    OPS_DOCS,
    dense_scores,
    filters={'tenant': 'acme', 'freshness': 'current'},
    candidate_k=3,
    final_k=2,
)
print('final:', [(doc.doc_id, round(score, 3)) for doc, score in results])
print('trace:', trace)
assert 'globex-secret' not in trace.fused_ids
assert 'acme-stale' not in trace.fused_ids


## 4. Evaluate the route, not one impressive answer

A hybrid route is justified only if it improves the labelled question distribution at an acceptable cost. Track candidate recall and MRR before answer generation, then separately evaluate context quality, citations, faithfulness, latency, and cost. Include identifier, paraphrase, compound, stale, no-answer, and cross-tenant cases.


In [ ]:
queries = [exact_query, paraphrase_query]
labels = [{'acme-runbook'}, {'acme-runbook'}]
bm25_rankings = [[doc for doc, _ in BM25(visible).search(q, top_k=3)] for q in queries]
hybrid_rankings = [
    [doc for doc, _ in hybrid_retrieve(q, OPS_DOCS, dense_scores, filters={'tenant': 'acme', 'freshness': 'current'})[0]]
    for q in queries
]
print('BM25 metrics:', ranking_metrics(bm25_rankings, labels))
print('hybrid metrics:', ranking_metrics(hybrid_rankings, labels))


## 5. Failure drills and checkpoint

1. Remove `acme-runbook` from the dense scores. Can reranking recover it if BM25 also misses it?
2. Change the tenant filter to `globex`. Prove Acme IDs never reach a candidate trace.
3. Increase `candidate_k` and record whether quality improves enough to justify the extra reranking work.
4. Add a policy with a synonym of `unauthorized`; test lexical, dense, and hybrid on a labelled case.
5. Which trace fields tell you whether an unsupported answer started as a source, filter, candidate, fusion, rerank, or generation failure?

### References

- Cormack, Clarke, and Buettcher, [Reciprocal Rank Fusion](https://cormack.uwaterloo.ca/cormacksigir09-rrf.pdf)
- [Sentence Transformers semantic search](https://www.sbert.net/examples/sentence_transformer/applications/semantic-search/README.html)
- [Qdrant hybrid and multi-stage queries](https://qdrant.tech/documentation/search/hybrid-queries/)
- [Qdrant hybrid reranking](https://qdrant.tech/documentation/advanced-tutorials/reranking-hybrid-search/)
